 SDXL_Research
Caption, train, compare and generate SDXL LoRAs on the Colab free T4.

Run **Setup** once per session. Re-runs skip finished steps.

In [ ]:
#@title Setup: repo + shared venv (idempotent)
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# venv is built by the system Python; the GPU work later runs in venv subprocesses,
# so this kernel never imports torch and never holds VRAM.
env = {**os.environ, "PYTHONPATH": str(REPO / "src")}
subprocess.run([sys.executable, "-m", "sdxlr.bootstrap"], env=env, check=True)

# Caption
Writes one `.txt` per image. Captions describe clothing, pose, background and
lighting, and skip face and body, so the LoRA learns those from the pixels.
Run with **Preview 3** first, check the captions, then run the full set.

In [ ]:
#@title Caption a dataset (Qwen3-VL / JoyCaption)
import subprocess
IMAGES_DIR = "/content/dataset"  #@param {type:"string"}
CAPTIONS_DIR = ""  #@param {type:"string"}
MODEL = "qwen"  #@param ["qwen", "qwen-2b", "joy"]
TRIGGER = "ohwx_person"  #@param {type:"string"}
PROMPT = ""  #@param {type:"string"}
PREVIEW = 3  #@param {type:"integer"}
OVERWRITE = False  #@param {type:"boolean"}

cmd = ["/content/venv/bin/python", "-m", "sdxlr.caption", "--images", IMAGES_DIR,
       "--model", MODEL, "--trigger", TRIGGER, "--limit", str(PREVIEW)]
if CAPTIONS_DIR: cmd += ["--out", CAPTIONS_DIR]
if PROMPT: cmd += ["--prompt", PROMPT]
if OVERWRITE: cmd += ["--overwrite"]
subprocess.run(cmd, cwd="/content/SDXL_Research", check=True)